## Launching model on dataset and collecting output

### Imports

In [11]:
from datasets import load_dataset
from tqdm import tqdm

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import Runnable, RunnableConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import Any, List

import torch
import sys

sys.path.append("../../../services/")
from index import Index
from process_funcs import retrieve_answer_token_index
from metric_funcs import calculate_entropy, calculate_norm_entropy

torch.random.manual_seed(42)

### Preparation

In [ ]:
ITERATIONS_COUNT = 12000
TOP_K = 50
LOWER_PROB_LIMIT = 1e-8
LOWER_LOGIT_LIMIT = -1000
UPPER_LOGIT_LIMIT = 1000

In [13]:
device = torch.device('cuda:5' if torch.cuda.is_available() else 'cpu')
print(device)
print(torch.cuda.get_device_properties(device))

cuda:5
_CudaDeviceProperties(name='NVIDIA H100 NVL', major=9, minor=0, total_memory=95248MB, multi_processor_count=132, uuid=d4d3fa02-fdea-80f5-9082-0157b1423027, L2_cache_size=60MB)


In [14]:
dataset = load_dataset("ehovy/race", "high")['train']

In [15]:
model_id = "mistralai/Mistral-7B-Instruct-v0.3"
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token_id = tokenizer.eos_token_id
model = model.to(device)
model.eval()

Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00, 43.74it/s]


MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): MistralRMSNorm((4096,), eps=1e-0

In [ ]:
class LLMInterface(Runnable):    
    def __init__(self, model, tokenizer, device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')):
        self.model = model
        self.tokenizer = tokenizer
        self.hook_handles = []
        self.device = device

        self.model = self.model.to(device)

    def invoke(        
        self,
        messages: List[Any],
        config: RunnableConfig | None = None,
        **kwargs: Any,
        ):
        hf_messages = [
            {"role": "system", "content": messages.messages[0].content},
            {"role": "user", "content": messages.messages[1].content},
        ]
                
        inputs = self.tokenizer.apply_chat_template(
            hf_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        model_inputs = self.tokenizer(
            inputs,
            return_tensors="pt"
        ).to(self.model.device)
        
        
        self.model.eval()
        with torch.no_grad():
            response = self.model.generate(
                **model_inputs,
                max_new_tokens=512,
                output_scores=True,
                output_attentions=True,
                return_dict_in_generate=True,
                do_sample=True,
                temperature=0.7,
                repetition_penalty=1.05,
                top_p=0.8,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        token_data = []
        generated_ids = response.sequences[:, model_inputs["input_ids"].shape[1]:].squeeze(0)
        for i, logits in enumerate(response.scores):
            logits = logits.squeeze(0)
            probs = torch.softmax(logits, dim=-1) 
            
            top_logits, top_tok_ids = torch.topk(logits, TOPK, dim=-1)
            top_logits = torch.clamp(top_logits, min=LOWER_LOGIT_LIMIT, max=UPPER_LOGIT_LIMIT)
            
            top_probs, _ = torch.topk(probs, TOPK, dim=-1)
            top_probs = top_probs.cpu()
            
            top_tokens = []
            for j in range(top_tok_ids.shape[-1]):
                top_tokens.append(self.tokenizer.decode(top_tok_ids[j].item()))
                
            token_id = generated_ids[i]
            token_data.append({
                "token": self.tokenizer.decode(token_id),
                "prob": probs[token_id].cpu().item(),
                "logit": logits[token_id].cpu().item(),
                "top_tokens": top_tokens,
                "top_logits": top_logits,
                "top_probs": top_probs,
            })   

        output_attetntions = [
            torch.stack(
                token_attention
            ).squeeze(1).clamp(min=LOWER_PROB_LIMIT).cpu()
            for token_attention in response.attentions[1:]
        ]

        attn_entropy = [
            calculate_entropy(x)
            for x in output_attetntions
        ]
        
        norm_attn_entropy = [
            calculate_norm_entropy(x)
            for x in output_attetntions
        ]
        
        return {
            "input_text": self.tokenizer.decode(response.sequences[:, :model_inputs["input_ids"].shape[1]].cpu()[0], skip_special_tokens=True),
            "output_text": self.tokenizer.decode(response.sequences[:, model_inputs["input_ids"].shape[1]:].cpu()[0], skip_special_tokens=True),
            "score_data": token_data,
            "attention_entropy": attn_entropy,
            "norm_attention_entropy": norm_attn_entropy
        }   

In [17]:
chat_llm = LLMInterface(
    model=model,
    tokenizer=tokenizer,
    device=device
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are an expert in answering multiple choice questions. 
            You will be given an article with question corresponding to it and answer options. Follow these rules strictly:
            
            1. Select the correct option number between 0 and 3
            2. Return your response as SINGLE token, only ONE number between 0 and 3

            Follow this output format:
            OPTION_NUMBER
            
            Example:
            2 

            Answer the following question about article:
            """,
        ),
        (
            "human",
            """
            Article: 
            {input_article}
            
            Question: 
            {input_question}
    
            Options:
            {input_options}
            """,
        ),
    ]
)


llm_chain = prompt | chat_llm

In [18]:
len(dataset)

62445

### Running model on dataset and collecting results


In [19]:
def filter_func(model_response, right_answer):
    token_data = model_response["score_data"][
        retrieve_answer_token_index(
            model_response["score_data"]
        )
    ]
    return right_answer in token_data["top_tokens"]

In [20]:
# launching model

ERROR_LIMIT = 500

index = Index(f"../../../index_data/mistral0.3_RACE_attn_cropped_{ITERATIONS_COUNT}")
index.clear()

correct_count = 0
progress_bar = tqdm(
    enumerate(dataset.select(range(ITERATIONS_COUNT))),
    total=ITERATIONS_COUNT,
    desc="Scoring RACE",
)

for iter, data in progress_bar:
    for error_count in range(ERROR_LIMIT + 1):
        try:
            response = llm_chain.invoke(
                {
                    "input_article": dataset[iter]["article"],
                    "input_question": dataset[iter]["question"],
                    "input_options": [
                        f"{i}:  {x}"
                        for i, x in enumerate(dataset[iter]["options"])
                    ],
                }
            )
            
            response["dataset_elem"] = dataset[iter]

            if filter_func(response, str(ord(data["answer"]) - ord('A'))):
                if response["output_text"][-1] == str(
                    ord(data["answer"]) - ord('A')
                ):
                    correct_count += 1

                progress_bar.set_postfix(
                    {"Current accuracy": str(correct_count / (iter + 1))}
                )
                
                response = {
                    "iteration": iter,
                    **response
                }

                index.save_data(response, iter, logging=False)
                break
            
            if error_count == ERROR_LIMIT:
                raise Exception("Error limit exceeded")
            
        except Exception as e:
            if (error_count + 1) % 500 == 0:
                print(f"Error on iteration {iter + 1}\nError count: {error_count}\nMessage: {e}")
            if error_count == ERROR_LIMIT:
                print("Error limit exceeded")

File is deleted: ../../../index_data/mistral0.3_RACE_attn_cropped_12000_index.pkl
File is deleted: ../../../index_data/mistral0.3_RACE_attn_cropped_12000_data.pkl
Index is cleared successfully.


Scoring RACE:   0%|          | 0/12000 [00:00<?, ?it/s]

Scoring RACE:  93%|█████████▎| 11202/12000 [56:32<13:02,  1.02it/s, Current accuracy=0.7056512811356129] 

Error on iteration 11202
Error count: 499
Message: CUDA error: unspecified launch failure
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Error limit exceeded


Scoring RACE:  93%|█████████▎| 11203/12000 [56:33<12:49,  1.04it/s, Current accuracy=0.7056512811356129]

...

Error on iteration 11998
Error count: 499
Message: CUDA error: unspecified launch failure
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Error limit exceeded


Scoring RACE: 100%|█████████▉| 11999/12000 [1:09:54<00:01,  1.07s/it, Current accuracy=0.7056512811356129]

Error on iteration 11999
Error count: 499
Message: CUDA error: unspecified launch failure
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Error limit exceeded


Scoring RACE: 100%|██████████| 12000/12000 [1:09:55<00:00,  2.86it/s, Current accuracy=0.7056512811356129]

Error on iteration 12000
Error count: 499
Message: CUDA error: unspecified launch failure
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Error limit exceeded


In [21]:
print(len(index))

11201
